# Restriction site scan

讀取 `../tables/assembled_scan.csv`，對每條 **full `Sequence`** 掃描下列限制酶切位。

- 同時掃正股與反股（非回文的 **Eco31I (GGTCTC)** 才需要，回文酶兩股相同）。
- 注意：BG5 / BG3 兩端背景是固定的，落在背景內的切位會出現在**每一條**序列。

In [ ]:
from pathlib import Path
import pandas as pd

# 限制酶辨識位點 (5'->3')
RE_SITES = {
    'BamHI':   'GGATCC',
    'BcuI':    'ACTAGT',   # = SpeI
    'BglII':   'AGATCT',
    'Eco31I':  'GGTCTC',   # = BsaI, 非回文 -> 連反股 GAGACC 一起掃
    'EcoRI':   'GAATTC',
    'HindIII': 'AAGCTT',
    'KpnI':    'GGTACC',
    'MluI':    'ACGCGT',
    'NcoI':    'CCATGG',
    'NdeI':    'CATATG',
    'NheI':    'GCTAGC',
    'NotI':    'GCGGCCGC',
    'PstI':    'CTGCAG',
    'SacI':    'GAGCTC',
    'SalI':    'GTCGAC',
    'SmaI':    'CCCGGG',
    'VspI':    'ATTAAT',   # = AseI
    'XbaI':    'TCTAGA',
    'XhoI':    'CTCGAG',
}

_comp = str.maketrans('ACGT', 'TGCA')
def revcomp(s):
    return s.translate(_comp)[::-1]

def find_sites(seq, site):
    '''回傳 site 在 seq 的所有起始位置 (正反股, 0-indexed)。'''
    seq = seq.upper()
    positions = []
    for p in {site, revcomp(site)}:
        start = 0
        while True:
            i = seq.find(p, start)
            if i == -1:
                break
            positions.append(i)
            start = i + 1
    return sorted(set(positions))

In [ ]:
# Add CSV/XLSX files here.
INPUT_FILES = [
    '../tables/assembled_scan.csv',
]

SEQUENCE_COL = 'Sequence'
OUTPUT_DIR = Path('../tables/RE_scan_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

def read_input_table(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    raise ValueError(f'Unsupported file type: {path}')

In [ ]:
def scan_one_file(path):
    path = Path(path)
    df = read_input_table(path)

    if SEQUENCE_COL not in df.columns:
        raise ValueError(f'{path} missing column: {SEQUENCE_COL}')

    enz_cols = list(RE_SITES.keys())
    res = df.copy()
    for enz, site in RE_SITES.items():
        res[enz] = [len(find_sites(s, site)) for s in df[SEQUENCE_COL].astype(str)]

    res['total_sites'] = res[enz_cols].sum(axis=1)
    res['clean'] = res['total_sites'] == 0
    res['source_file'] = path.name

    summary = pd.DataFrame({
        'source_file': path.name,
        'enzyme': enz_cols,
        'site': [RE_SITES[e] for e in enz_cols],
        'n_seqs_with_site': [(res[e] > 0).sum() for e in enz_cols],
        'total_occurrences': [res[e].sum() for e in enz_cols],
    }).sort_values('n_seqs_with_site', ascending=False)

    absent = summary[summary['total_occurrences'] == 0].copy()

    res.to_csv(OUTPUT_DIR / f'{path.stem}_RE_scan.csv', index=False)
    summary.to_csv(OUTPUT_DIR / f'{path.stem}_RE_summary.csv', index=False)
    absent.to_csv(OUTPUT_DIR / f'{path.stem}_absent_RE_sites.csv', index=False)

    clean_n = int(res['clean'].sum())
    clean_pct = res['clean'].mean()
    print(f'{path.name}: no-cut sequences {clean_n}/{len(res)} = {clean_pct:.1%}')

    return res, summary, absent

In [ ]:
all_results = []
all_summaries = []
all_absent = []

for file in INPUT_FILES:
    res_i, summary_i, absent_i = scan_one_file(file)
    all_results.append(res_i)
    all_summaries.append(summary_i)
    all_absent.append(absent_i)

combined = pd.concat(all_results, ignore_index=True)
combined_summary = pd.concat(all_summaries, ignore_index=True)
combined_absent = pd.concat(all_absent, ignore_index=True)

combined.to_csv(OUTPUT_DIR / 'ALL_files_RE_scan.csv', index=False)
combined_summary.to_csv(OUTPUT_DIR / 'ALL_files_RE_summary.csv', index=False)
combined_absent.to_csv(OUTPUT_DIR / 'ALL_files_absent_RE_sites_by_file.csv', index=False)

# Keep these names for later cells / quick inspection.
df = all_results[0]
res = all_results[0]
summary = all_summaries[0]
enz_cols = list(RE_SITES.keys())

combined_summary

In [ ]:
# RE sites absent in every input file.
absent_sets = [
    set(summary_i.loc[summary_i['total_occurrences'] == 0, 'enzyme'])
    for summary_i in all_summaries
]

common_absent_enzymes = sorted(set.intersection(*absent_sets)) if absent_sets else []

common_absent_RE_sites = pd.DataFrame({
    'enzyme': common_absent_enzymes,
    'site': [RE_SITES[e] for e in common_absent_enzymes],
    'status': 'absent_in_all_files',
})

common_absent_RE_sites.to_csv(OUTPUT_DIR / 'common_absent_RE_sites.csv', index=False)
print(f'common absent RE sites: {len(common_absent_RE_sites)}')
print(f'saved -> {OUTPUT_DIR / "common_absent_RE_sites.csv"}')

common_absent_RE_sites

In [ ]:
# 查單條序列的詳細切位 (酶 -> 位置清單)
def site_detail(seq):
    seq = str(seq)
    return {enz: find_sites(seq, site) for enz, site in RE_SITES.items() if find_sites(seq, site)}

site_detail(df['Sequence'].iloc[0])